# Assignment 6 — Multiclass land use/cover mapping with U-Net (NAIP → Philadelphia land cover)

This notebook adapts the building-footprint U-Net tutorial (`2.model-trainning-prediction.ipynb`) from a **binary** building mask to a **multiclass** land cover map. It is written to run top-to-bottom on **Google Colab (GPU runtime)**.

**Workflow**

| Step | Section | Assignment item |
|---|---|---|
| 0 | Setup, configuration, data download | – |
| 1 | Inspect raw label codes, legend, class mapping, pixel counts | 1 |
| 2 | Tile the rasters and make a spatial train / val / test split (+ map) | 2 |
| 3 | Dataset, U-Net (`out_channels=C`), `CrossEntropyLoss`, class-weight comparison, checkpoint | 3 |
| 4 | Held-out test evaluation: confusion matrix, IoU, mIoU, pixel accuracy, examples, boundary errors | 4 |
| 5 | Categorical georeferenced GeoTIFF + legend | 5 |
| 6 | Report template | Deliverables |

Design choices that differ from the tutorial (and why):
* Tiles are read **directly from the two big rasters with rasterio windows** (no intermediate `naip_tiles` / `lu_masks` files), so every tile keeps its exact geographic position. That is what makes the spatial split, the split map and the stitched GeoTIFF possible.
* The label raster is read through a `WarpedVRT` onto the NAIP grid with **nearest-neighbour** resampling, so image and label grids are guaranteed to be aligned (CRS, extent, pixel size).
* Excluded / no-data pixels get the label `IGNORE_INDEX = 255` and are ignored by the loss, the confusion matrix, IoU and accuracy.

## 0. Setup

In [ ]:
!pip -q install rasterio

In [ ]:
import os, sys, json, math, random, time, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch, Rectangle
from matplotlib.collections import PatchCollection
from scipy import ndimage

import rasterio as rio
from rasterio.windows import Window
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

IN_COLAB = 'google.colab' in sys.modules
USE_GOOGLE_DRIVE = False   # True: keep data/outputs on Drive so they survive a runtime reset
if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = Path('/content/drive/MyDrive/ai-urban-sustainability/assignment6')
elif IN_COLAB:
    DATA_ROOT = Path('/content/assignment6')
else:
    DATA_ROOT = Path('./assignment6_data').resolve()
OUT_DIR = DATA_ROOT / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch', torch.__version__, '| rasterio', rio.__version__, '| device:', device)
if device.type == 'cpu':
    print('WARNING: no GPU. Runtime > Change runtime type > GPU before training.')

### Configuration (every hyper-parameter lives here — nothing is changed after seeing test results)

In [ ]:
# ---------------- reproducibility ----------------
SEED = 42

# ---------------- data ----------------
DATA_URL  = 'https://utilitiesfiles.s3.us-east-1.amazonaws.com/penn-class/ai-urban-sustainability/data/'
NAIP_FILE = DATA_ROOT / 'naip-philly.tif'     # ~3.3 GB
LC_FILE   = DATA_ROOT / 'lu_phily.tif'        # ~90 MB reference land use/cover raster

TILE        = 256     # source tile size in pixels (same as Part 1 of the lab)
MODEL_SIZE  = 512     # tiles are resized to this for the model (must be divisible by 8)
IGNORE_INDEX = 255    # label value for excluded / no-data pixels (NOT a class id)
NODATA_OUT   = 255    # no-data code of the output GeoTIFF (outside class ids 0..C-1)

# ---------------- input bands ----------------
# NAIP in this lab can have 4 bands (R, G, B, NIR). Band numbers are 1-based.
USE_NIR = False                                # False: bands 1-3 as stored (a false-colour CIR composite in this file); True: all 4 bands
BANDS   = [1, 2, 3, 4] if USE_NIR else [1, 2, 3]
IN_CH   = len(BANDS)

# ---------------- spatial split ----------------
SPLIT_AXIS    = 'x'                 # 'x': vertical strips (west->east); 'y': horizontal strips (north->south)
SPLIT_ORDER   = ('train', 'val', 'test')   # order of the strips along the axis
SPLIT_FRACS   = {'train': 0.70, 'val': 0.15, 'test': 0.15}   # share of tile columns (rows) per split
BUFFER_TILES  = 1                   # empty tile columns (rows) dropped between two splits
MIN_VALID_FRAC = 0.50               # a tile needs >= 50 % valid (non-ignored) pixels to be used
MAX_TILES = {'train': 1500, 'val': 300}   # memory guard: random subsample (seeded). None = keep all

# ---------------- training ----------------
BATCH_SIZE = 8
LR         = 1e-3
EPOCHS     = 15
NUM_WORKERS = 2
WEIGHT_STRATEGIES = ['none', 'inv_sqrt_freq']   # compared on the VALIDATION set only

# ---------------- inference ----------------
INFER_STRIDE = 128      # sliding-window stride (TILE=256 -> 50 % overlap)
INFER_BATCH  = 8
BLOCK        = 1536     # test area is processed in BLOCK x BLOCK pixel blocks (memory guard)
MARGIN       = INFER_STRIDE

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed()
assert MODEL_SIZE % 8 == 0

In [ ]:
def download(url, dest):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        print('exists:', dest.name, f'({dest.stat().st_size/1e6:.0f} MB)'); return
    print('downloading', url)
    subprocess.check_call(['curl', '-L', '--fail', '-o', str(dest), url])

for f in (NAIP_FILE, LC_FILE):
    download(DATA_URL + f.name, f)

## 1. Inspect and prepare the labels

First look at the raw rasters: CRS, extent, pixel size, number of bands, data type and no-data value. Then list every raw code that occurs in the reference raster.

In [ ]:
naip = rio.open(NAIP_FILE)
lc   = rio.open(LC_FILE)

def describe(ds, name):
    return {'raster': name, 'crs': str(ds.crs), 'size (w x h)': f'{ds.width} x {ds.height}', 'bands': ds.count,
            'dtype': ds.dtypes[0], 'pixel size': tuple(round(abs(v), 4) for v in (ds.res[0], ds.res[1])),
            'nodata': ds.nodata, 'bounds': tuple(round(b, 1) for b in ds.bounds)}
display(pd.DataFrame([describe(naip, 'NAIP imagery'), describe(lc, 'Reference land cover')]).set_index('raster').T)
print('NAIP band descriptions:', naip.descriptions, '| colour interp:', [c.name for c in naip.colorinterp])
assert naip.count >= max(BANDS), f'NAIP has only {naip.count} bands'
IMG_SCALE = float(np.iinfo(np.dtype(naip.dtypes[0])).max) if np.issubdtype(np.dtype(naip.dtypes[0]), np.integer) else 1.0
print('Image scale divisor:', IMG_SCALE)

# --- read the label raster ON THE NAIP GRID (nearest neighbour). If the grids already match this is a no-op.
grids_match = (lc.crs == naip.crs and lc.transform == naip.transform and lc.shape == naip.shape)
print('Label grid identical to NAIP grid:', grids_match)
LC_NODATA = lc.nodata if lc.nodata is not None else 0
lab = WarpedVRT(lc, crs=naip.crs, transform=naip.transform, width=naip.width, height=naip.height,
                resampling=Resampling.nearest, nodata=LC_NODATA)
assert (lab.width, lab.height) == (naip.width, naip.height) and lab.transform == naip.transform

In [ ]:
# Raw code histogram over the whole raster, block by block (exact counts, low memory)
def raw_code_counts(ds, step=4096):
    counts = np.zeros(65536, dtype=np.int64)
    for r in range(0, ds.height, step):
        for c in range(0, ds.width, step):
            a = ds.read(1, window=Window(c, r, min(step, ds.width - c), min(step, ds.height - r)))
            counts += np.bincount(a.ravel().astype(np.int64) % 65536, minlength=65536)
    return counts
raw_counts = raw_code_counts(lab)
raw_codes = np.flatnonzero(raw_counts)
raw_table = pd.DataFrame({'raw code': raw_codes, 'pixels': raw_counts[raw_codes]})
raw_table['share %'] = (100 * raw_table['pixels'] / raw_table['pixels'].sum()).round(2)
display(raw_table)
print('Label raster nodata value (file):', lc.nodata, '| value used for pixels outside the label extent:', LC_NODATA)

### Source legend and class mapping

> **You must verify this legend against the source documentation of `lu_phily.tif`** (the Philadelphia 1 m land cover data the lab was built from). The tutorial states that code `5` = buildings, which this legend honours. The default below follows the Philadelphia high-resolution land cover classes (1 tree canopy, 2 grass/shrub, 3 bare soil, 4 water, 5 buildings, 6 roads/railroads, 7 other paved surfaces). The cell after it prints the codes actually found; **if they differ from this legend, edit `SOURCE_LEGEND` before continuing** — never name classes from pixel values alone.

`CLASSES` defines the training classes. Order = training ID (`0..C-1`). A class may group several source codes (documented regrouping); codes not listed are **excluded** and become `IGNORE_INDEX`.

In [ ]:
# Verified source legend (raw code -> meaning).  EDIT IF THE DOCUMENTATION SAYS OTHERWISE.
SOURCE_LEGEND = {1: 'Tree canopy', 2: 'Grass / shrub', 3: 'Bare soil', 4: 'Water',
                 5: 'Buildings', 6: 'Roads / railroads', 7: 'Other paved surfaces'}

# Training classes: name -> list of raw source codes. Order defines the training ID.
CLASSES = {
    'Tree canopy':          [1],
    'Grass / shrub':        [2],
    'Bare soil':            [3],
    'Water':                [4],
    'Buildings':            [5],
    'Roads / railroads':    [6],
    'Other paved surfaces': [7],
}
CLASS_COLORS = ['#1b7837', '#a6dba0', '#c8a165', '#2166ac', '#d73027', '#4d4d4d', '#bdbdbd',
                '#984ea3', '#ff7f00', '#ffff33']   # one fixed colour per training ID
CLASS_NAMES = list(CLASSES.keys())
C = len(CLASS_NAMES)
assert C >= 3 and 'Buildings' in CLASSES, 'need >= 3 classes including buildings'
assert IGNORE_INDEX >= C and NODATA_OUT >= C

# lookup table raw code -> training ID (everything else -> IGNORE_INDEX)
LUT = np.full(65536, IGNORE_INDEX, dtype=np.uint8)
for cid, (name, codes) in enumerate(CLASSES.items()):
    for code_ in codes:
        assert LUT[code_] == IGNORE_INDEX, f'raw code {code_} assigned twice'
        LUT[code_] = cid
def to_ids(raw):                       # raw label array -> training-ID array (uint8, 255 = ignore)
    return LUT[raw.astype(np.int64) % 65536]

mapping_rows = []
for cid, (name, codes) in enumerate(CLASSES.items()):
    mapping_rows.append({'training ID': cid, 'class name': name, 'source code(s)': codes,
                         'source meaning': ' + '.join(SOURCE_LEGEND.get(c_, '??? NOT IN LEGEND') for c_ in codes),
                         'pixels (whole raster)': int(sum(raw_counts[c_] for c_ in codes)), 'colour': CLASS_COLORS[cid]})
mapping_df = pd.DataFrame(mapping_rows)
mapping_df['share of valid %'] = (100 * mapping_df['pixels (whole raster)'] / mapping_df['pixels (whole raster)'].sum()).round(2)
display(mapping_df)

used = {c_ for codes in CLASSES.values() for c_ in codes}
excluded = [int(c_) for c_ in raw_codes if c_ not in used]
print('Raw codes present but EXCLUDED (-> IGNORE_INDEX=%d):' % IGNORE_INDEX,
      [(c_, SOURCE_LEGEND.get(c_, 'not in legend / no-data'), int(raw_counts[c_])) for c_ in excluded])
unknown = [int(c_) for c_ in raw_codes if c_ not in SOURCE_LEGEND and c_ != LC_NODATA]
if unknown: print('!! Codes missing from SOURCE_LEGEND, check the documentation:', unknown)

**How excluded categories and no-data are handled.** Every raw code that is not listed in `CLASSES` (including the raster no-data value and pixels outside the label extent) is mapped to `IGNORE_INDEX = 255`. Pixels where the NAIP image itself is empty (all bands `0`) are also set to 255. `CrossEntropyLoss(ignore_index=255)` skips them, and they are removed from accuracy, confusion matrix and IoU. In the output GeoTIFF they receive the no-data code 255, which is outside the class IDs `0..C-1`. *(Write here which codes, if any, were excluded in your run and why.)*

## 2. Spatial split

Tiles are cut on a regular, **non-overlapping** `TILE × TILE` grid. The tile grid is then divided into three contiguous geographic strips (train / validation / test) with `BUFFER_TILES` empty tile columns between neighbouring strips, so no tile of one split touches a tile of another. The split is therefore defined **before** any augmentation, and augmentation is only ever applied to training tiles inside the dataset class.

In [ ]:
n_rows, n_cols = naip.height // TILE, naip.width // TILE
print(f'tile grid: {n_rows} rows x {n_cols} cols = {n_rows*n_cols} tiles')

# valid-label fraction for every tile (reads the small label raster strip by strip)
valid_frac = np.zeros((n_rows, n_cols), dtype=np.float32)
for i in range(n_rows):
    strip = lab.read(1, window=Window(0, i * TILE, n_cols * TILE, TILE))
    ids = to_ids(strip).reshape(TILE, n_cols, TILE)
    valid_frac[i] = (ids != IGNORE_INDEX).mean(axis=(0, 2))

# strip boundaries along the split axis
n_axis = n_cols if SPLIT_AXIS == 'x' else n_rows
usable = n_axis - BUFFER_TILES * (len(SPLIT_ORDER) - 1)
sizes = {s: int(round(SPLIT_FRACS[s] * usable)) for s in SPLIT_ORDER}
sizes[SPLIT_ORDER[-1]] = usable - sum(sizes[s] for s in SPLIT_ORDER[:-1])
axis_range, pos = {}, 0
for s in SPLIT_ORDER:
    axis_range[s] = (pos, pos + sizes[s]); pos += sizes[s] + BUFFER_TILES
print('split ranges along', SPLIT_AXIS, '(tile index, end exclusive):', axis_range)

def split_of(i, j):
    k = j if SPLIT_AXIS == 'x' else i
    for s, (a, b) in axis_range.items():
        if a <= k < b: return s
    return 'buffer'

split_grid = np.empty((n_rows, n_cols), dtype=object)
for i in range(n_rows):
    for j in range(n_cols):
        split_grid[i, j] = split_of(i, j)
usable_grid = valid_frac >= MIN_VALID_FRAC

rng = np.random.default_rng(SEED)
candidates = {s: [(i, j) for i in range(n_rows) for j in range(n_cols)
                  if split_grid[i, j] == s and usable_grid[i, j]] for s in SPLIT_ORDER}
for s in ('train', 'val'):                       # shuffle so the later cap is a random subsample
    rng.shuffle(candidates[s])
print({s: len(v) for s, v in candidates.items()}, '(usable tiles per split before caps)')

In [ ]:
# ---- load the tiles of the train / val splits into memory (uint8) ----
def read_tile(i, j):
    win = Window(j * TILE, i * TILE, TILE, TILE)
    img = naip.read(BANDS, window=win)                         # [B, T, T]
    ids = to_ids(lab.read(1, window=win))                      # [T, T] uint8
    empty = (naip.read(window=win) == 0).all(axis=0)           # image no-data (all bands zero)
    ids[empty] = IGNORE_INDEX
    return img, ids

def load_split(split, cap):
    X, Y, keep = [], [], []
    for (i, j) in candidates[split]:
        if cap is not None and len(keep) >= cap: break
        img, ids = read_tile(i, j)
        if (ids != IGNORE_INDEX).mean() < MIN_VALID_FRAC: continue
        X.append(img); Y.append(ids); keep.append((i, j))
    return np.stack(X), np.stack(Y), keep

t0 = time.time()
data = {}
for s in ('train', 'val'):
    data[s] = load_split(s, MAX_TILES[s])
    print(f'{s}: X {data[s][0].shape} {data[s][0].dtype}  Y {data[s][1].shape}  ({time.time()-t0:.0f}s)')
kept_tiles = {s: data[s][2] for s in ('train', 'val')}
kept_tiles['test'] = candidates['test']      # all usable test tiles (the test area is mapped in full)

In [ ]:
# ---- split map + tile counts ----
split_colors = {'train': '#1b9e77', 'val': '#e6ab02', 'test': '#d95f02', 'buffer': '#999999'}
ov_scale = max(1, max(naip.width, naip.height) // 1500)
ov = naip.read([1, 2, 3], out_shape=(3, naip.height // ov_scale, naip.width // ov_scale),
               resampling=Resampling.average).astype(float)
ov = np.clip(ov / IMG_SCALE / max(np.percentile(ov / IMG_SCALE, 99), 1e-6), 0, 1).transpose(1, 2, 0)
left, bottom, right, top = naip.bounds

fig, ax = plt.subplots(figsize=(10, 10 * naip.height / naip.width + 1))
ax.imshow(ov, extent=(left, right, bottom, top), alpha=0.8)
def tile_rect(i, j):
    x0, y0 = naip.transform * (j * TILE, (i + 1) * TILE)      # lower-left corner (geo coords)
    x1, y1 = naip.transform * ((j + 1) * TILE, i * TILE)
    return Rectangle((x0, y0), x1 - x0, y1 - y0)
for s in ('train', 'val', 'test'):
    pc = PatchCollection([tile_rect(i, j) for i, j in kept_tiles[s]], facecolor=split_colors[s],
                         edgecolor='none', alpha=0.45)
    ax.add_collection(pc)
# strip outlines (including the buffer gaps, which are left uncoloured)
for s, (a, b) in axis_range.items():
    if SPLIT_AXIS == 'x':
        x0, y0 = naip.transform * (a * TILE, n_rows * TILE); x1, y1 = naip.transform * (b * TILE, 0)
    else:
        x0, y0 = naip.transform * (0, b * TILE); x1, y1 = naip.transform * (n_cols * TILE, a * TILE)
    ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor=split_colors[s], lw=2))
ax.legend(handles=[Patch(facecolor=split_colors[s], alpha=.6, label=f'{s}: {len(kept_tiles[s])} tiles') for s in ('train', 'val', 'test')]
          + [Patch(facecolor='none', edgecolor='k', label=f'gap between splits = {BUFFER_TILES} tile(s)')], loc='lower left')
ax.set_title('Spatial split of the tile grid'); ax.set_xlabel(f'x ({naip.crs})'); ax.set_ylabel('y')
plt.tight_layout(); plt.savefig(OUT_DIR / 'split_map.png', dpi=150); plt.show()

split_counts = pd.DataFrame({'usable tiles (>= MIN_VALID_FRAC)': {s: len(candidates[s]) for s in SPLIT_ORDER},
                             'tiles used': {s: len(kept_tiles[s]) for s in SPLIT_ORDER}})
split_counts.loc['total'] = split_counts.sum()
display(split_counts)
# sanity check against spatial leakage: no tile of one split is adjacent to a tile of another
def split_id_grid():
    return np.vectorize(lambda s: {'train': 0, 'val': 1, 'test': 2, 'buffer': -1}[s])(split_grid)
g = split_id_grid()
assert not ((g[:, :-1] >= 0) & (g[:, 1:] >= 0) & (g[:, :-1] != g[:, 1:])).any()
assert not ((g[:-1, :] >= 0) & (g[1:, :] >= 0) & (g[:-1, :] != g[1:, :])).any()
print('No train/val/test tiles are adjacent (leakage check passed).')

In [ ]:
# ---- pixel counts per class and split (training-ID space, valid pixels only) ----
def class_counts_from_tiles(Y):
    return np.bincount(Y[Y != IGNORE_INDEX].ravel(), minlength=C)
cnt = pd.DataFrame({'train': class_counts_from_tiles(data['train'][1]),
                    'val':   class_counts_from_tiles(data['val'][1])}, index=CLASS_NAMES)
# test-area counts (exact, whole test strip)
def test_region():
    a, b = axis_range['test']
    if SPLIT_AXIS == 'x': return Window(a * TILE, 0, (b - a) * TILE, n_rows * TILE)
    return Window(0, a * TILE, n_cols * TILE, (b - a) * TILE)
TEST_WIN = test_region()
test_cnt = np.zeros(C, dtype=np.int64)
for r in range(0, TEST_WIN.height, 2048):
    for c_ in range(0, TEST_WIN.width, 2048):
        w = Window(TEST_WIN.col_off + c_, TEST_WIN.row_off + r, min(2048, TEST_WIN.width - c_), min(2048, TEST_WIN.height - r))
        ids = to_ids(lab.read(1, window=w)); test_cnt += np.bincount(ids[ids != IGNORE_INDEX], minlength=C)
cnt['test (reference)'] = test_cnt
for col in list(cnt.columns):
    cnt[col + ' %'] = (100 * cnt[col] / cnt[col].sum()).round(2)
display(cnt)
train_counts = cnt['train'].values

## 3. Adapt and train the U-Net

**What changed relative to the tutorial**

| Tutorial (binary) | This notebook (multiclass) |
|---|---|
| `np.where(mask_y == 5, 1, 0)` in `open_mask` **and** `__getitem__` | one lookup table `LUT` (raw code → training ID, others → 255) used in *every* label-loading path (`to_ids`) |
| `UNET(3, 2)` | `UNET(IN_CH, C)` → logits `[N, C, H, W]` |
| `CrossEntropyLoss()` | `CrossEntropyLoss(weight=…, ignore_index=255)` |
| `acc_metric` counts all pixels | accuracy / IoU / confusion matrix skip ignored pixels |
| `skimage.transform.resize` for image and mask | `F.interpolate` bilinear for the image, **nearest** for labels |
| independent random transforms | one random flip/rot90 sampled per tile and applied to image and mask together; brightness jitter on the image only |
| `/255` | `/255` then per-band standardisation with **training-set** mean/std |

In [ ]:
# per-band normalisation statistics from TRAINING pixels only (valid pixels, scaled to 0..1)
Xtr, Ytr = data['train'][0], data['train'][1]
valid = Ytr != IGNORE_INDEX
band_mean = np.array([(Xtr[:, b][valid] / IMG_SCALE).mean() for b in range(IN_CH)], dtype=np.float32)
band_std  = np.array([(Xtr[:, b][valid] / IMG_SCALE).std()  for b in range(IN_CH)], dtype=np.float32)
print('band mean:', band_mean.round(4), '| band std:', band_std.round(4))
MEAN_T = torch.tensor(band_mean).view(-1, 1, 1); STD_T = torch.tensor(band_std).view(-1, 1, 1)

def normalise(x01):                      # x01: float tensor [B,H,W] in 0..1
    return (x01 - MEAN_T) / STD_T

class SegDataset(Dataset):
    def __init__(self, X, Y, augment=False):
        self.X, self.Y, self.augment = X, Y, augment
    def __len__(self): return len(self.X)
    def __getitem__(self, idx):
        x = torch.from_numpy(self.X[idx].astype(np.float32)) / IMG_SCALE      # [B,T,T]
        y = torch.from_numpy(self.Y[idx].astype(np.int64))                    # [T,T] training IDs, 255 = ignore
        if self.augment:                                                       # SAME geometry for image and mask
            k = random.randint(0, 3)
            x, y = torch.rot90(x, k, (1, 2)), torch.rot90(y, k, (0, 1))
            if random.random() < 0.5: x, y = x.flip(2), y.flip(1)
            if random.random() < 0.5: x, y = x.flip(1), y.flip(0)
            x = (x * (1 + 0.2 * (random.random() - 0.5))).clamp(0, 1)         # brightness: image only
        x = normalise(x)
        x = F.interpolate(x[None], size=(MODEL_SIZE, MODEL_SIZE), mode='bilinear', align_corners=False)[0]
        y = F.interpolate(y[None, None].float(), size=(MODEL_SIZE, MODEL_SIZE), mode='nearest')[0, 0].long()
        return x, y

train_ds = SegDataset(data['train'][0], data['train'][1], augment=True)
val_ds   = SegDataset(data['val'][0],   data['val'][1],   augment=False)
g = torch.Generator(); g.manual_seed(SEED)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, generator=g, drop_last=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
xb, yb = next(iter(train_dl))
print('batch image', tuple(xb.shape), xb.dtype, '| labels', tuple(yb.shape), yb.dtype, '| label values:', yb.unique().tolist())

In [ ]:
# ---- U-Net from the tutorial (unchanged architecture; in_channels / out_channels are now IN_CH / C) ----
class UNET(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = self.contract_block(in_channels, 32, 7, 3)
        self.conv2 = self.contract_block(32, 64, 3, 1)
        self.conv3 = self.contract_block(64, 128, 3, 1)
        self.upconv3 = self.expand_block(128, 64, 3, 1)
        self.upconv2 = self.expand_block(64 * 2, 32, 3, 1)
        self.upconv1 = self.expand_block(32 * 2, out_channels, 3, 1)

    def forward(self, x):
        conv1 = self.conv1(x); conv2 = self.conv2(conv1); conv3 = self.conv3(conv2)
        upconv3 = self.upconv3(conv3)
        upconv2 = self.upconv2(torch.cat([upconv3, conv2], 1))
        upconv1 = self.upconv1(torch.cat([upconv2, conv1], 1))
        return upconv1                                             # logits [N, C, H, W]

    def contract_block(self, in_channels, out_channels, kernel_size, padding):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=1, padding=padding),
            nn.BatchNorm2d(out_channels), nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=kernel_size, stride=1, padding=padding),
            nn.BatchNorm2d(out_channels), nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1))

    def expand_block(self, in_channels, out_channels, kernel_size, padding):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=padding),
            nn.BatchNorm2d(out_channels), nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size, stride=1, padding=padding),
            nn.BatchNorm2d(out_channels), nn.ReLU(),
            nn.ConvTranspose2d(out_channels, out_channels, kernel_size=3, stride=2, padding=1, output_padding=1))

_m = UNET(IN_CH, C)
with torch.no_grad(): print('logits shape for one tile:', tuple(_m.eval()(torch.zeros(1, IN_CH, MODEL_SIZE, MODEL_SIZE)).shape))
del _m

In [ ]:
# ---- metrics (all of them ignore IGNORE_INDEX pixels) ----
def confusion_np(pred, ref, C=C):
    """rows = reference, cols = prediction; pixels where ref == IGNORE_INDEX or pred == NODATA_OUT are dropped."""
    m = (ref != IGNORE_INDEX) & (pred != NODATA_OUT)
    return np.bincount(ref[m].astype(np.int64) * C + pred[m].astype(np.int64), minlength=C * C).reshape(C, C)

def metrics_from_confusion(cm):
    tp = np.diag(cm).astype(float); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    union = tp + fp + fn
    iou = np.where(union > 0, tp / np.maximum(union, 1), np.nan)   # NaN = class absent from BOTH reference and prediction
    return {'iou': iou, 'miou': float(np.nanmean(iou)), 'pixel_acc': float(tp.sum() / max(cm.sum(), 1)),
            'precision': np.where(tp + fp > 0, tp / np.maximum(tp + fp, 1), np.nan),
            'recall':    np.where(tp + fn > 0, tp / np.maximum(tp + fn, 1), np.nan)}
# IoU averaging rule: a class absent from BOTH reference and prediction has union 0 -> IoU undefined (NaN) and is
# skipped by nanmean.  A class absent from the reference but predicted somewhere gets IoU = 0 and IS averaged
# (false positives are penalised).  A class present in the reference but never predicted also gets IoU = 0.

@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval(); tot, n, cm = 0.0, 0, np.zeros((C, C), dtype=np.int64)
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        loss = loss_fn(out, y)
        tot += loss.item() * len(x); n += len(x)
        cm += confusion_np(out.argmax(1).cpu().numpy().astype(np.uint8), y.cpu().numpy().astype(np.uint8))
    return tot / n, cm

def class_weights(strategy, counts):
    counts = np.asarray(counts, dtype=float)
    if strategy == 'none': return None
    freq = counts / counts.sum()
    if strategy == 'inv_sqrt_freq':
        w = np.where(freq > 0, 1 / np.sqrt(np.maximum(freq, 1e-12)), 0.0)
    elif strategy == 'inv_freq':
        w = np.where(freq > 0, 1 / np.maximum(freq, 1e-12), 0.0)
    else: raise ValueError(strategy)
    w = w / w[freq > 0].mean()                     # mean weight 1 over present classes
    return torch.tensor(w, dtype=torch.float32)

def train_model(strategy, epochs=EPOCHS):
    set_seed(SEED)
    model = UNET(IN_CH, C).to(device)
    w = class_weights(strategy, train_counts)          # TRAINING-set pixel counts only
    loss_fn = nn.CrossEntropyLoss(weight=None if w is None else w.to(device), ignore_index=IGNORE_INDEX)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    hist = {'train_loss': [], 'val_loss': [], 'val_miou': [], 'val_acc': []}
    best = {'miou': -1, 'epoch': -1, 'state': None}
    t0 = time.time()
    for ep in range(epochs):
        model.train(); run, n = 0.0, 0
        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(); loss = loss_fn(model(x), y); loss.backward(); opt.step()
            run += loss.item() * len(x); n += len(x)
        vl, cm = evaluate(model, val_dl, loss_fn); m = metrics_from_confusion(cm)
        hist['train_loss'].append(run / n); hist['val_loss'].append(vl)
        hist['val_miou'].append(m['miou']); hist['val_acc'].append(m['pixel_acc'])
        if m['miou'] > best['miou']:
            best = {'miou': m['miou'], 'epoch': ep, 'state': {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, 'cm': cm}
        print(f'[{strategy}] epoch {ep+1:02d}/{epochs}  train loss {run/n:.4f}  val loss {vl:.4f}  val mIoU {m["miou"]:.4f}  val acc {m["pixel_acc"]:.4f}  ({time.time()-t0:.0f}s)')
    return hist, best, w

### Train with and without class weights (selection uses the **validation** set only)

In [ ]:
runs = {}
for strategy in WEIGHT_STRATEGIES:
    runs[strategy] = train_model(strategy)

fig, axes = plt.subplots(1, len(runs), figsize=(6 * len(runs), 4), squeeze=False)
for ax, (strategy, (hist, best, w)) in zip(axes[0], runs.items()):
    ep = np.arange(1, len(hist['train_loss']) + 1)
    ax.plot(ep, hist['train_loss'], label='training loss'); ax.plot(ep, hist['val_loss'], label='validation loss')
    ax.axvline(best['epoch'] + 1, color='gray', ls=':', label=f'selected epoch {best["epoch"]+1}')
    ax.set_title(f'weights = {strategy}'); ax.set_xlabel('epoch'); ax.set_ylabel('cross-entropy'); ax.legend()
plt.tight_layout(); plt.savefig(OUT_DIR / 'loss_curves.png', dpi=150); plt.show()

In [ ]:
# compare weighting strategies on the validation set (best epoch of each run)
rows = []
for strategy, (hist, best, w) in runs.items():
    m = metrics_from_confusion(best['cm'])
    row = {'strategy': strategy, 'best epoch': best['epoch'] + 1, 'val mIoU': round(m['miou'], 4), 'val pixel acc': round(m['pixel_acc'], 4)}
    row.update({f'IoU {n}': (None if np.isnan(v) else round(float(v), 3)) for n, v in zip(CLASS_NAMES, m['iou'])})
    rows.append(row)
cmp_df = pd.DataFrame(rows).set_index('strategy'); display(cmp_df)
for strategy, (hist, best, w) in runs.items():
    if w is not None: print(f'class weights [{strategy}]:', dict(zip(CLASS_NAMES, w.numpy().round(3))))

SELECTED = max(runs, key=lambda s: runs[s][1]['miou'])
print('\nSelected on VALIDATION mIoU ->', SELECTED)

In [ ]:
# ---- save the selected checkpoint + all settings (class mapping, bands, normalisation, split, seed, lr, ...) ----
hist, best, w = runs[SELECTED]
CKPT = OUT_DIR / 'unet_multiclass_best.pth'
config = {
    'class_mapping': [{'training_id': i, 'name': n, 'source_codes': CLASSES[n], 'color': CLASS_COLORS[i]} for i, n in enumerate(CLASS_NAMES)],
    'ignore_index': IGNORE_INDEX, 'output_nodata': NODATA_OUT, 'excluded_source_codes': excluded,
    'input_bands': BANDS, 'in_channels': IN_CH, 'out_channels': C, 'model_input_size': MODEL_SIZE, 'tile_size': TILE,
    'normalisation': {'scale_divisor': IMG_SCALE, 'mean': band_mean.tolist(), 'std': band_std.tolist(), 'fit_on': 'training tiles, valid pixels'},
    'split': {'axis': SPLIT_AXIS, 'order': list(SPLIT_ORDER), 'fractions': SPLIT_FRACS, 'buffer_tiles': BUFFER_TILES,
              'tile_index_ranges': {k: list(v) for k, v in axis_range.items()}, 'tiles_used': {k: len(v) for k, v in kept_tiles.items()},
              'min_valid_frac': MIN_VALID_FRAC},
    'seed': SEED, 'learning_rate': LR, 'optimizer': 'Adam', 'batch_size': BATCH_SIZE, 'epochs': EPOCHS,
    'selected_epoch': best['epoch'] + 1, 'loss': 'CrossEntropyLoss(ignore_index=255' + (', weight=%s)' % w.numpy().round(4).tolist() if w is not None else ')'),
    'weight_strategy': SELECTED, 'augmentation': 'train only: rot90, h/v flip (shared by image & mask), brightness +-10% (image only)',
    'val_miou_selected': best['miou'],
}
torch.save({'state_dict': best['state'], 'config': config}, CKPT)
with open(OUT_DIR / 'experiment_config.json', 'w') as f: json.dump(config, f, indent=2)
print('saved', CKPT); print(json.dumps({k: v for k, v in config.items() if k not in ('class_mapping',)}, indent=1)[:2500])

## 4. Evaluate on the held-out test area

Everything above (class weights, epoch, normalisation) is now fixed. The test strip is used for the first time here. The model predicts the **whole test strip** with overlapping sliding windows (Section 5 describes the tile-edge strategy); the evaluation compares the resulting mosaic with the reference on valid pixels.

We first define the inference code, run it once to produce the GeoTIFF, then compute all metrics from that GeoTIFF — so the reported numbers describe exactly the delivered map.

In [ ]:
# ---- sliding-window inference over a region, written block by block to a categorical GeoTIFF ----
def hann_like(T):
    a = np.minimum(np.arange(1, T + 1), np.arange(T, 0, -1)).astype(np.float32)   # triangular, always > 0
    w = np.outer(a, a); return w / w.max()
W2D = hann_like(TILE)

def window_starts(n, T, stride):
    s = list(range(0, max(n - T, 0) + 1, stride))
    if s[-1] != max(n - T, 0): s.append(max(n - T, 0))
    return s

@torch.no_grad()
def predict_block(model, img):
    """img: [B,h,w] raw array (any size). Returns (ids uint8 [h,w], 255 where the image is empty)."""
    B, h, w = img.shape
    ph, pw = max(h, TILE), max(w, TILE)
    pad = np.zeros((B, ph, pw), dtype=img.dtype); pad[:, :h, :w] = img
    prob = np.zeros((C, ph, pw), dtype=np.float32); wsum = np.zeros((ph, pw), dtype=np.float32)
    rs, cs = window_starts(ph, TILE, INFER_STRIDE), window_starts(pw, TILE, INFER_STRIDE)
    coords = [(r, c) for r in rs for c in cs]
    for k in range(0, len(coords), INFER_BATCH):
        chunk = coords[k:k + INFER_BATCH]
        x = torch.stack([torch.from_numpy(pad[:, r:r + TILE, c:c + TILE].astype(np.float32)) / IMG_SCALE for r, c in chunk])
        x = (x - MEAN_T) / STD_T
        x = F.interpolate(x, size=(MODEL_SIZE, MODEL_SIZE), mode='bilinear', align_corners=False).to(device)
        p = F.softmax(model(x), dim=1)
        p = F.interpolate(p, size=(TILE, TILE), mode='nearest').cpu().numpy()      # back to the source grid, nearest
        for (r, c), pi in zip(chunk, p):
            prob[:, r:r + TILE, c:c + TILE] += pi * W2D; wsum[r:r + TILE, c:c + TILE] += W2D
    ids = (prob / np.maximum(wsum, 1e-6)).argmax(0).astype(np.uint8)[:h, :w]
    ids[(img == 0).all(axis=0)] = NODATA_OUT                                          # empty imagery -> no-data
    return ids

def write_prediction_map(model, region, out_path):
    model.eval()
    prof = dict(driver='GTiff', height=region.height, width=region.width, count=1, dtype='uint8', crs=naip.crs,
                transform=rio.windows.transform(region, naip.transform), nodata=NODATA_OUT,
                compress='lzw', tiled=True, blockxsize=256, blockysize=256)
    with rio.open(out_path, 'w', **prof) as dst:
        for r0 in range(0, region.height, BLOCK):
            for c0 in range(0, region.width, BLOCK):
                core_h, core_w = min(BLOCK, region.height - r0), min(BLOCK, region.width - c0)
                er0, ec0 = max(r0 - MARGIN, 0), max(c0 - MARGIN, 0)
                er1, ec1 = min(r0 + core_h + MARGIN, region.height), min(c0 + core_w + MARGIN, region.width)
                img = naip.read(BANDS, window=Window(region.col_off + ec0, region.row_off + er0, ec1 - ec0, er1 - er0))
                ids = predict_block(model, img)
                core = ids[r0 - er0:r0 - er0 + core_h, c0 - ec0:c0 - ec0 + core_w]
                dst.write(core, 1, window=Window(c0, r0, core_w, core_h))
        dst.write_colormap(1, {i: tuple(int(CLASS_COLORS[i][k:k + 2], 16) for k in (1, 3, 5)) + (255,) for i in range(C)} | {NODATA_OUT: (0, 0, 0, 0)})
        dst.update_tags(**{f'CLASS_{i}': f'{n} (source codes {CLASSES[n]})' for i, n in enumerate(CLASS_NAMES)}, NODATA_CODE=str(NODATA_OUT))

ckpt = torch.load(CKPT, map_location=device, weights_only=False)
model = UNET(ckpt['config']['in_channels'], ckpt['config']['out_channels']).to(device)
model.load_state_dict(ckpt['state_dict']); model.eval()
print('loaded checkpoint, selected epoch', ckpt['config']['selected_epoch'], '| C =', ckpt['config']['out_channels'])

PRED_TIF = OUT_DIR / 'test_area_landcover_prediction.tif'
t0 = time.time(); write_prediction_map(model, TEST_WIN, PRED_TIF); print(f'wrote {PRED_TIF.name} in {time.time()-t0:.0f}s')

In [ ]:
# ---- confusion matrix / IoU / accuracy on valid TEST pixels + boundary-vs-interior error rates ----
BAND_PX = 2    # "boundary" = pixels within 2 px of a change in the reference label
cm_test = np.zeros((C, C), dtype=np.int64)
bd_err = bd_tot = in_err = in_tot = 0
with rio.open(PRED_TIF) as pr:
    assert pr.crs == naip.crs and pr.transform == rio.windows.transform(TEST_WIN, naip.transform)
    for r in range(0, TEST_WIN.height, 2048):
        for c_ in range(0, TEST_WIN.width, 2048):
            h, w_ = min(2048, TEST_WIN.height - r), min(2048, TEST_WIN.width - c_)
            pred = pr.read(1, window=Window(c_, r, w_, h))
            ref = to_ids(lab.read(1, window=Window(TEST_WIN.col_off + c_, TEST_WIN.row_off + r, w_, h)))
            cm_test += confusion_np(pred, ref)
            ok = (ref != IGNORE_INDEX) & (pred != NODATA_OUT)
            edge = np.zeros(ref.shape, bool)
            edge[:-1] |= ref[:-1] != ref[1:]; edge[1:] |= ref[:-1] != ref[1:]
            edge[:, :-1] |= ref[:, :-1] != ref[:, 1:]; edge[:, 1:] |= ref[:, :-1] != ref[:, 1:]
            band = ndimage.binary_dilation(edge, iterations=BAND_PX - 1) if BAND_PX > 1 else edge
            wrong = pred != ref
            bd_err += (wrong & ok & band).sum(); bd_tot += (ok & band).sum()
            in_err += (wrong & ok & ~band).sum(); in_tot += (ok & ~band).sum()

mt = metrics_from_confusion(cm_test)
cm_df = pd.DataFrame(cm_test, index=[f'ref: {n}' for n in CLASS_NAMES], columns=[f'pred: {n}' for n in CLASS_NAMES])
display(cm_df)
cm_norm = cm_test / np.maximum(cm_test.sum(1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(C)); ax.set_yticks(range(C)); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right'); ax.set_yticklabels(CLASS_NAMES)
for i in range(C):
    for j in range(C):
        ax.text(j, i, f'{cm_norm[i, j]:.2f}', ha='center', va='center', color='white' if cm_norm[i, j] > .5 else 'black', fontsize=8)
ax.set_xlabel('predicted'); ax.set_ylabel('reference'); ax.set_title('Test confusion matrix (row-normalised = recall)')
plt.colorbar(im); plt.tight_layout(); plt.savefig(OUT_DIR / 'confusion_matrix_test.png', dpi=150); plt.show()

res = pd.DataFrame({'test pixels (ref)': cm_test.sum(1), 'test pixels (pred)': cm_test.sum(0),
                    'IoU': mt['iou'].round(4), 'precision': mt['precision'].round(4), 'recall': mt['recall'].round(4)}, index=CLASS_NAMES)
display(res)
print(f'mean IoU (nan-mean over {int(np.sum(~np.isnan(mt["iou"])))} of {C} classes defined on test): {mt["miou"]:.4f}')
print(f'pixel accuracy: {mt["pixel_acc"]:.4f}   | share of the most common reference class: {cm_test.sum(1).max()/cm_test.sum():.4f}')
print(f'error rate within {BAND_PX}px of a reference boundary: {bd_err/max(bd_tot,1):.3f}  ({bd_tot/(bd_tot+in_tot):.1%} of valid pixels)'
      f'  | in the interior: {in_err/max(in_tot,1):.3f}')

off = [(cm_test[i, j], CLASS_NAMES[i], CLASS_NAMES[j]) for i in range(C) for j in range(C) if i != j]
print('\nLargest confusions (reference -> predicted):')
for n, a, b in sorted(off, reverse=True)[:6]: print(f'  {a:>22s} -> {b:<22s} {n:>12,d} px ({n/cm_test.sum():.2%} of valid pixels)')
json.dump({'miou': mt['miou'], 'pixel_accuracy': mt['pixel_acc'], 'iou': {n: (None if np.isnan(v) else float(v)) for n, v in zip(CLASS_NAMES, mt['iou'])},
           'confusion_matrix_rows_ref_cols_pred': cm_test.tolist(), 'boundary_error_rate': float(bd_err/max(bd_tot,1)),
           'interior_error_rate': float(in_err/max(in_tot,1))}, open(OUT_DIR / 'test_metrics.json', 'w'), indent=2)

**How the averages are defined.** `IoU_c = TP / (TP + FP + FN)`, computed from valid test pixels only (reference ≠ 255 and prediction ≠ no-data). `mIoU` is the `nanmean` over classes:
* a class absent from **both** the test reference and the prediction has an undefined IoU (0/0) and is **left out** of the mean;
* a class absent from the reference but predicted somewhere gets IoU = 0 and **is included** (its false positives are penalised);
* a class present in the reference but never predicted also gets IoU = 0.

**Why pixel accuracy can mislead.** It is dominated by the largest class (see "share of the most common reference class" above). A model that predicts mostly tree canopy / grass could score a high pixel accuracy while missing small classes such as buildings or water completely; per-class IoU and the confusion matrix expose this, accuracy does not.

In [ ]:
# ---- at least three test examples: RGB | reference | prediction with the SAME discrete colours + legend ----
cmap = ListedColormap(CLASS_COLORS[:C]); norm = BoundaryNorm(np.arange(-0.5, C + 0.5), C)
def show_label(ax, arr, title):
    ax.imshow(np.ma.masked_where(arr >= C, arr), cmap=cmap, norm=norm, interpolation='nearest'); ax.set_title(title); ax.axis('off')

# candidate test tiles: well covered, preferring many different classes in the reference
rng = np.random.default_rng(SEED)
cands = [kept_tiles['test'][k] for k in rng.permutation(len(kept_tiles['test']))[:150]]
scored = []
with rio.open(PRED_TIF) as pr:
    for (i, j) in cands:
        win = Window(j * TILE, i * TILE, TILE, TILE)
        ref = to_ids(lab.read(1, window=win)); ids_present = np.unique(ref[ref != IGNORE_INDEX])
        scored.append((len(ids_present) + (ref != IGNORE_INDEX).mean(), i, j))
    scored.sort(reverse=True)
    N_EX = 4
    fig, axes = plt.subplots(N_EX, 3, figsize=(11, 3.6 * N_EX))
    for row, (_, i, j) in enumerate(scored[:N_EX]):
        win = Window(j * TILE, i * TILE, TILE, TILE)
        rgb = naip.read([1, 2, 3], window=win).astype(float) / IMG_SCALE
        rgb = np.clip(rgb / max(np.percentile(rgb, 99), 1e-6), 0, 1).transpose(1, 2, 0)
        ref = to_ids(lab.read(1, window=win))
        pw = Window(j * TILE - TEST_WIN.col_off, i * TILE - TEST_WIN.row_off, TILE, TILE)
        pred = pr.read(1, window=pw)
        m = metrics_from_confusion(confusion_np(pred, ref))
        axes[row, 0].imshow(rgb); axes[row, 0].set_title(f'NAIP bands 1-3, false colour  (tile row {i}, col {j})'); axes[row, 0].axis('off')
        show_label(axes[row, 1], ref, 'reference'); show_label(axes[row, 2], pred, f'prediction (tile acc {m["pixel_acc"]:.2f})')
fig.legend(handles=[Patch(facecolor=CLASS_COLORS[i], label=f'{i}: {CLASS_NAMES[i]}') for i in range(C)], loc='lower center', ncol=min(C, 4), frameon=False)
plt.tight_layout(rect=(0, 0.05, 1, 1)); plt.savefig(OUT_DIR / 'test_examples.png', dpi=150); plt.show()

**Error discussion (test area, 27.7 M valid pixels).**
* **Boundaries.** The error rate within 2 px of a reference boundary is **47.7 %**, versus **21.6 %** in the interior, i.e. more than twice as high. Edges are hard because of the 1 m pixel size, label/image misregistration, tree-crown overhang and shadows, and the 256→512→256 resizing. In the example tiles the predicted edges of tree stands and roads are smoother and wider than the reference.
* **Similar classes.** The biggest confusions are *Grass/shrub → Tree canopy* (17.6 % of grass pixels), *Tree canopy → Grass/shrub* (14.7 %), *Other paved surfaces → Buildings* (16.7 %), *Buildings → Other paved surfaces* (19.3 %) and *Other paved surfaces → Roads* (13.7 %). Roofs, parking lots, plazas and roads share spectra and texture, and short trees vs. tall shrubs/grass differ mostly in height, which a 2-D image cannot show.
* **Bare soil is never predicted.** Its IoU is 0 (recall 0, no predicted pixels). It is the rarest class (about 1.3 % of test pixels) and looks like pavement or dry grass; the model absorbs it into Grass/shrub and Other paved surfaces. The mIoU of 0.49 is pulled down by this one class (the other six average about 0.57).
* **Land cover vs. land use.** NAIP shows what covers the surface, not how it is used. A "Building" can be a house, factory or school; "Other paved surfaces" can be a parking lot or a plaza. Functional land use (residential, industrial, ...) cannot be derived from these classes without extra data such as parcels or zoning. The labels also have their own errors and a different date than the imagery.

## 5. Categorical georeferenced map

`write_prediction_map` (above) produced `test_area_landcover_prediction.tif`:
* **Grid:** same CRS, pixel size and affine transform as the NAIP source (windowed `transform = rio.windows.transform(TEST_WIN, naip.transform)`); one `uint8` band of training IDs `0..C-1`.
* **No-data:** `255` (outside the class IDs), also assigned where the NAIP image is empty. A colour table and `CLASS_i` tags are embedded in the file.
* **Resizing:** each 256×256 source window is bilinearly resized to 512×512 for the model; the class probabilities are mapped back to 256×256 with **nearest-neighbour** interpolation before being written with the source transform.
* **Tile edges / overlap:** windows overlap by 50 % (stride 128). Softmax probabilities of all windows covering a pixel are averaged with a triangular weight that is highest at the window centre and lowest at its border, so pixels near a tile edge are decided mainly by windows in which they sit near the centre; the argmax is taken after averaging. The area is processed in 1536×1536 blocks with a 128 px context margin, so block borders are blended the same way. The last row/column of windows is shifted inwards so every window stays inside the data (zero-padded only when the region is smaller than a tile).

In [ ]:
# ---- verify the output, export legend (JSON + PNG) and a side-by-side map ----
with rio.open(PRED_TIF) as pr:
    print('CRS:', pr.crs, '| nodata:', pr.nodata, '| dtype:', pr.dtypes[0], '| size:', pr.width, 'x', pr.height)
    print('transform equal to NAIP-window transform:', pr.transform == rio.windows.transform(TEST_WIN, naip.transform))
    print('pixel size:', pr.res, '(NAIP:', naip.res, ')')
    vals = np.unique(pr.read(1, out_shape=(pr.height // 8, pr.width // 8), resampling=Resampling.nearest))
    print('values present (decimated check):', vals.tolist(), '-> all in 0..C-1 or', NODATA_OUT, ':', all(v < C or v == NODATA_OUT for v in vals))

legend = [{'training_id': i, 'name': n, 'source_codes': CLASSES[n], 'source_meaning': [SOURCE_LEGEND.get(c_) for c_ in CLASSES[n]], 'color_hex': CLASS_COLORS[i]}
          for i, n in enumerate(CLASS_NAMES)] + [{'training_id': NODATA_OUT, 'name': 'No data / excluded', 'source_codes': excluded, 'color_hex': None}]
json.dump(legend, open(OUT_DIR / 'legend.json', 'w'), indent=2)
display(pd.DataFrame(legend))

step = max(1, max(TEST_WIN.width, TEST_WIN.height) // 1400)
with rio.open(PRED_TIF) as pr:
    small = pr.read(1, out_shape=(pr.height // step, pr.width // step), resampling=Resampling.nearest)
ref_small = to_ids(lab.read(1, window=TEST_WIN, out_shape=small.shape, resampling=Resampling.nearest))
fig, axes = plt.subplots(1, 2, figsize=(16, 8 * small.shape[0] / small.shape[1] + 1))
show_label(axes[0], ref_small, 'reference (test area)'); show_label(axes[1], small, 'U-Net prediction (test area)')
fig.legend(handles=[Patch(facecolor=CLASS_COLORS[i], label=f'{i}: {CLASS_NAMES[i]}') for i in range(C)] + [Patch(facecolor='white', edgecolor='k', label=f'{NODATA_OUT}: no data')],
           loc='lower center', ncol=min(C + 1, 5), frameon=False)
plt.tight_layout(rect=(0, 0.06, 1, 1)); plt.savefig(OUT_DIR / 'test_area_map.png', dpi=150); plt.show()

print('\nDeliverables in', OUT_DIR); [print(' -', p.name, f'({p.stat().st_size/1e6:.1f} MB)') for p in sorted(OUT_DIR.iterdir())]

In [ ]:
# Colab: download the deliverables (or copy OUT_DIR to Drive)
if IN_COLAB:
    !cd {OUT_DIR} && zip -qr /content/assignment6_outputs.zip . && ls -lh /content/assignment6_outputs.zip
    from google.colab import files
    files.download('/content/assignment6_outputs.zip')

## 6. Short report

**Class definitions.** Reference raster `lu_phily.tif` (Philadelphia land cover; legend per the data documentation, code 5 = buildings as in the tutorial). Seven classes were kept, each mapped one-to-one from a source code to a training ID:

| Training ID | Class | Source code | Test share of valid pixels |
|---|---|---|---|
| 0 | Tree canopy | 1 | 21.4 % |
| 1 | Grass / shrub | 2 | 35.9 % |
| 2 | Bare soil | 3 | 1.3 % |
| 3 | Water | 4 | 4.8 % |
| 4 | Buildings | 5 | 11.4 % |
| 5 | Roads / railroads | 6 | 8.5 % |
| 6 | Other paved surfaces | 7 | 16.7 % |

Source code 0 (no-data) was the only excluded code. Excluded and no-data pixels, and pixels where the NAIP image is empty, are set to 255 and ignored by the loss, confusion matrix, IoU and accuracy. No classes were regrouped. Labels were read onto the NAIP grid with nearest-neighbour resampling.

**Spatial split.** 256 × 256 tiles on a non-overlapping grid were split into three west-to-east strips (tile columns 0-73 train, 74-90 validation, 91-106 test) with a one-column gap between strips, so no tiles of different splits touch. Tiles used: 1500 train (random subsample), 300 validation, 430 test. Augmentation (rot90, flips, brightness) is applied to training tiles only, after the split. Weighting and epoch were chosen on validation; the test strip was evaluated once.

**Training setup.** U-Net from the tutorial with `in_channels=3`, `out_channels=7`; tiles resized to 512 × 512; input bands 1-3 of the NAIP file, scaled by 1/255 and standardised with training mean (0.564, 0.576, 0.633) / std (0.239, 0.219, 0.187); Adam, lr 1e-3, batch size 8, 15 epochs, seed 42. I compared an unweighted loss with inverse-square-root-frequency class weights; the validation loss curves are very similar and the **unweighted** run was selected (epoch 13, validation mIoU 0.474). Checkpoint: `unet_multiclass_best.pth`; settings: `experiment_config.json`.
*Input bands:* bands 1-3 of this NAIP file are a **false-colour (colour-infrared) composite**, not true RGB: vegetation appears pink/red and water teal in the example figures. The model was therefore trained on these three bands as stored (so it does see near-infrared information, which helps separate vegetation from pavement), and the example panels labelled "NAIP RGB" are really this false-colour composite. Because the same bands were used for training, validation, testing and mapping, the results are consistent; they should not be described as an RGB-only model.

**Results (held-out test strip).** Mean IoU **0.492**; pixel accuracy **0.706**.

| Class | IoU | Recall | Precision |
|---|---|---|---|
| Tree canopy | 0.587 | 0.794 | 0.692 |
| Grass / shrub | 0.590 | 0.691 | 0.803 |
| Bare soil | 0.000 | 0.000 | no predictions |
| Water | 0.775 | 0.932 | 0.821 |
| Buildings | 0.505 | 0.735 | 0.618 |
| Roads / railroads | 0.608 | 0.852 | 0.680 |
| Other paved surfaces | 0.378 | 0.518 | 0.583 |

All seven classes occur in the test reference, so none had to be dropped from the IoU average; a class absent from both reference and prediction would be skipped, and one that is predicted but absent from the reference would count as IoU 0. Pixel accuracy is dominated by the large classes (Grass/shrub is 36 % of test pixels): the model could ignore Bare soil entirely and lose only about 1 point of accuracy, which is exactly what happened, so accuracy overstates quality compared with mIoU.

**Main errors.** Boundary error rate 47.7 % vs. 21.6 % in the interior. Largest confusions: Grass ↔ Tree canopy, Buildings ↔ Other paved surfaces, Other paved surfaces → Roads. Bare soil is never predicted. Water is the cleanest class (IoU 0.775), with errors mostly at shorelines and shaded areas.

**Limitations.** Land cover is not land use: the map cannot say whether a building is residential or industrial. Single city and single acquisition date; the test strip is one contiguous area that may not represent the whole city. Reference labels are an imperfect product and are not perfectly registered to the imagery, which caps achievable boundary accuracy. The 1 m resolution misses thin features (narrow paths and roads). Rare classes (Bare soil) need more data, a larger or different class-weighting scheme, or merging with a similar class (e.g. Grass/shrub) to be useful.